# Deutsch's Algorithm
**Notebook:** Deutsch0 — single-qubit oracle, statevector analysis

# Standard Qiskit imports
import qiskit as qk
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector        # exact statevector simulation (no noise)
from qiskit.visualization import plot_bloch_multivector  # Bloch sphere visualisation
from math import pi

In [ ]:
import qiskit as qk
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
from math import pi

### Consider the Deusch problem with q Qubit $f(x) \rightarrow 0 \textrm{ or } 1 \textrm{ , where } x \textrm{ is } 0 \textrm{ or } 1$. Define 4 oracles :

# Oracle 1 — constant f(0)=f(1)=0
# Implementation: identity (empty circuit). No gates needed because
# flipping the ancilla is never triggered; phase kickback leaves input qubit unchanged.
oracle1 = qk.QuantumCircuit(2, name='oracle1')
oracle1.draw()

In [4]:
oracle1=qk.QuantumCircuit(2, name='oracle1')
oracle1.draw()

q_0: 
     
q_1:

# Oracle 2 — balanced f(0)=0, f(1)=1
# Implementation: CNOT with q0 (input) as control and q1 (ancilla) as target.
# When input is |0⟩, ancilla is unchanged; when input is |1⟩, ancilla is flipped.
# Via phase kickback: the ancilla |−⟩ = (|0⟩−|1⟩)/√2 picks up a phase of −1
# only when the input is |1⟩, encoding f(x)=x into the relative phase.
oracle2 = qk.QuantumCircuit(2, name='oracle2')
oracle2.cx(0, 1)   # CNOT: control=q0 (input), target=q1 (ancilla)
oracle2.draw()

In [5]:
oracle2=qk.QuantumCircuit(2, name='oracle2')
oracle2.cx(0,1)
oracle2.draw()

q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

# Oracle 3 — balanced f(0)=1, f(1)=0  (the NOT of oracle2)
# Implementation: CNOT followed by X on the ancilla.
# CNOT flips ancilla when input=|1⟩, then X flips ancilla unconditionally.
# Net effect: ancilla flipped when input=|0⟩ (f(0)=1), not flipped when input=|1⟩ (f(1)=0).
# Phase kickback encodes the complemented balanced function into the input qubit phase.
oracle3 = qk.QuantumCircuit(2, name='oracle3')
oracle3.cx(0, 1)   # CNOT: flip ancilla if input is |1⟩
oracle3.x(1)       # X on ancilla: inverts the condition (now flips when input is |0⟩)
oracle3.draw()

In [6]:
oracle3=qk.QuantumCircuit(2, name='oracle3')
oracle3.cx(0,1)
oracle3.x(1)
oracle3.draw()

q_0: ──■───────
     ┌─┴─┐┌───┐
q_1: ┤ X ├┤ X ├
     └───┘└───┘

# Oracle 4 — constant f(0)=f(1)=1
# Implementation: X on the ancilla only (no dependence on the input qubit).
# The ancilla is flipped regardless of the input, so f(x)=1 for all x.
# Because the flip is unconditional, phase kickback applies the same global phase
# to both |0⟩ and |1⟩ input states — global phases are unobservable, leaving
# the input qubit unchanged (constant behaviour confirmed by measurement giving |0⟩).
oracle4 = qk.QuantumCircuit(2, name='oracle4')
oracle4.x(1)   # X on ancilla q1: flip unconditionally (input-independent → constant)
oracle4.draw()

In [7]:
oracle4=qk.QuantumCircuit(2, name='oracle4')
oracle4.x(1)
oracle4.draw()

q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

# Run Deutsch's algorithm for all 4 oracles using exact statevector simulation.
# Expected results: oracles 1 & 4 → CONSTANT (input qubit collapses to |0⟩)
#                   oracles 2 & 3 → BALANCED (input qubit collapses to |1⟩)

oracles = [oracle1, oracle2, oracle3, oracle4]
labels  = [
    'oracle1 (constant:  f(0)=f(1)=0)',
    'oracle2 (balanced:  f(0)=0, f(1)=1)',
    'oracle3 (balanced:  f(0)=1, f(1)=0)',
    'oracle4 (constant:  f(0)=f(1)=1)',
]

for oracle, label in zip(oracles, labels):
    circ = qk.QuantumCircuit(2)

    # Step 1: prepare ancilla in |1⟩ so it becomes |−⟩ after the next Hadamard
    circ.x(1)

    # Step 2: apply Hadamard to both qubits
    # q0 (input):   |0⟩ → |+⟩ = (|0⟩+|1⟩)/√2
    # q1 (ancilla): |1⟩ → |−⟩ = (|0⟩−|1⟩)/√2
    circ.h([0, 1])

    # Step 3: apply the oracle — encodes f(x) via phase kickback onto q0
    circ.append(oracle, [0, 1])

    # Step 4: apply Hadamard to the input qubit only
    # Interference: if f constant → q0 returns to |0⟩; if f balanced → q0 → |1⟩
    circ.h(0)

    # Step 5: compute marginal probabilities for q0 via the statevector
    sv    = Statevector(circ)
    probs = sv.probabilities_dict(qargs=[0])   # marginalise over input qubit q0

    # Interpret result: highest-probability basis state tells us constant vs balanced
    result = '|0⟩  → CONSTANT' if max(probs, key=probs.get) == '0' else '|1⟩  → BALANCED'
    print(f"{label}\n  input qubit: {result}  (probs={probs})\n")

In [ ]:
# Visualise the final statevector on the Bloch sphere for oracle4 (constant f=1).
# After the full Deutsch circuit, q0 should point to the |0⟩ pole (north),
# confirming the constant classification. The ancilla q1 ends in |−⟩ (equator).
circ4 = qk.QuantumCircuit(2)
circ4.x(1)            # ancilla → |1⟩
circ4.h([0, 1])       # both qubits into superposition
circ4.append(oracle4, [0, 1])  # oracle4: X on ancilla (constant f=1)
circ4.h(0)            # final Hadamard on input qubit → resolves to |0⟩ for constant
sv = Statevector(circ4)
plot_bloch_multivector(sv)   # left sphere = q0 (input), right sphere = q1 (ancilla)

In [ ]:
# Correctness assertions — verify that the statevector analysis agrees with theory.
# Constant oracles (1, 4) must yield '0' on q0; balanced oracles (2, 3) must yield '1'.
expected = ['constant', 'balanced', 'balanced', 'constant']

for oracle, exp in zip(oracles, expected):
    circ = qk.QuantumCircuit(2)
    circ.x(1)               # ancilla → |1⟩
    circ.h([0, 1])          # superposition on both qubits
    circ.append(oracle, [0, 1])  # query the oracle once
    circ.h(0)               # interference on input qubit

    sv    = Statevector(circ)
    probs = sv.probabilities_dict(qargs=[0])  # marginal over input qubit

    # Map most-probable outcome to a classification label
    detected = 'constant' if max(probs, key=probs.get) == '0' else 'balanced'

    assert detected == exp, f"Expected {exp}, got {detected}"

print("All assertions passed: Deutsch algorithm works correctly!")

In [ ]:
# Assertions: verify constant oracles give |0⟩, balanced give |1⟩
expected = ['constant', 'balanced', 'balanced', 'constant']
for oracle, exp in zip(oracles, expected):
    circ = qk.QuantumCircuit(2)
    circ.x(1)
    circ.h([0, 1])
    circ.append(oracle, [0, 1])
    circ.h(0)
    sv = Statevector(circ)
    probs = sv.probabilities_dict(qargs=[0])
    detected = 'constant' if max(probs, key=probs.get) == '0' else 'balanced'
    assert detected == exp, f"Expected {exp}, got {detected}"
print("All assertions passed: Deutsch algorithm works correctly!")